# Load Data from an Intan RHD file.
### Header information (and saved data, if present) will be stored in the *result* variable. You'll also be able to specify a channel's name to plot its recorded data as an example of accessing data from *result*.

In [1]:
import glob
import pandas as pd
import numpy as np
import os

In [2]:
def make_fig_checkStimulus(result, name, strts, stps):
    import matplotlib.pyplot as plt
    
    f = plt.figure()
    plt.plot(result['t_board_adc'][0:12*30000], result['board_adc_data'][0][0:12*30000])
    plt.savefig(path_for_fig + name + '_firstWNRepeat.png')
    plt.close()
    
    f = plt.figure()
    plt.plot(result['board_dig_in_data'][0])
    plt.plot(strts, np.ones(len(strts)), 'ok')
    plt.plot(stps, np.ones(len(stps)), 'or')
    plt.savefig(path_for_fig + name + '_strtStop_align.png')
    plt.close()
    
    ## Making sure they are aligned at the index I have pulled out!
    f = plt.figure()
    for ss, st in zip(strts, stps):
        plt.plot(result['board_adc_data'][0][ss-10000:st+10000])
    plt.xlim(7500,15000)
    plt.savefig(path_for_fig + name + '_WN_align.png')
    plt.close()

In [3]:
# Load the background Python functions that allow for data loading and plotting

%run importrhdutilities.py

In [4]:
rhd_files = glob.glob("../dataFolder/1_RHD_Files/*.rhd")
rhd_files

path_for_fig = "../dataFolder/Figures/CheckStimulusWhenLoadingRHD/"
os.makedirs(path_for_fig, exist_ok=True)

In [13]:
# pull channel number on which data was sorted for each recording
channel_dict = pd.read_csv("../dataFolder/2_Spikesorted_Files/ReadMe.txt", delimiter = ' ')
channel_dict = channel_dict.set_index('FileID')['channelSorted'].to_dict()

In [6]:
# path for explorting data 
out_path = "./../dataFolder/3_ReadRHD_files/"
os.makedirs(out_path, exist_ok=True)

### Name of file to load. To load a different file, move it to the same directory as this notebook, and change the following 'filename' variable to point the name of your file.

In [ ]:
rhd_files[]

In [14]:
for filename in rhd_files[4:]:
    print(f'Processing: {filename}')
    result, data_present = load_file(filename)
    name = os.path.basename(filename).split('.')[0]
    
    #Once a file is loaded, all of its information is stored in the result variable. 
    #If further analysis of recorded data is desired, this is the variable that should be used.
    
    # Check amplifier voltage range
    print(f"Amplifier voltage range: {result['amplifier_data'].min():.2f} to {result['amplifier_data'].max():.2f} µV")
    print(f"shape of amplifier data is {result['amplifier_data'].shape}")

    # Check ADC voltage range
    print(f"ADC voltage range: {result['board_adc_data'][0].min():.4f} to {result['board_adc_data'][0].max():.4f} V")
    
    # Pulling out the up and down index of the digital input
    digiIn = result['board_dig_in_data'][0]
    strts = digiIn
    strts = [idx for idx, v in enumerate(digiIn) if (v) & (not digiIn[idx-1])]
    stps = [idx for idx, v in enumerate(digiIn) if (not v) & (digiIn[idx-1])]
    
    # make figures for sanity check
    make_fig_checkStimulus(result, name, strts, stps)
    
    # specify the channel used for sorting data - A-012 would be 11
    channel = channel_dict[name];
    
    ## export index values
    index = pd.DataFrame({'index_starts':strts,
                 'index_stops': stps})
    index.to_csv(out_path + name + '_index.csv', index = False)

    ## export raw data - stimulus and amplifier data
    RawData = pd.DataFrame({'stimulus': result['board_adc_data'][0]}) 
    RawData.to_csv(out_path + name + '_RawStimulusData.csv', index=False)
    
    amplifier_channel = pd.DataFrame({'raw_neural_trace': result['amplifier_data'][channel]})
    amplifier_channel.to_csv(out_path + name + '_rawNeuralData.csv', index = False)

Processing: ../dataFolder/1_RHD_Files\m29_wn2.rhd

Reading Intan Technologies RHD Data File, Version 1.5

Found 16 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 1 board ADC channel.
Found 1 board digital input channel.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

File contains 360.960 seconds of data.  Amplifiers were sampled at 30.00 kS/s.

Allocating memory for data...
Reading data from file...
10% done...
20% done...
30% done...
40% done...
50% done...
60% done...
70% done...
80% done...
90% done...
Parsing data...
Done!  Elapsed time: 32.4 seconds
Amplifier voltage range: -1949.22 to 1188.53 µV
shape of amplifier data is (16, 10828800)
ADC voltage range: 0.6300 to 2.2719 V
Processing: ../dataFolder/1_RHD_Files\m30_wn1.rhd

Reading Intan Technologies RHD Data File, Version 1.5

Found 16 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 1 board ADC channel.
Found